In [1]:
import re
import os
import sys
from lca_algebraic import initProject
import pandas as pd
import numpy as np
import xarray as xr
import lca_algebraic as agb
from lca_algebraic import *
from lca_algebraic.stats import * 
import bw2data, bw2io
import bw2calc
from premise import *
from itertools import zip_longest

In [2]:
sys.path.append('/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/') 
from utils import *
from utils.setup import *
from utils.getAct import *
from utils.utils import *
from utils.dlcia_functions import *

modify the setup.py file for reading Parameter excel sheet. here we use 'data/p_file/p_tech_perSSP_Y.xlsx' as the technology parameter file.  
Define your chosen methods in the setup.py file. Current methods are:  [('IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT'), ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
Mapping premise_remind_DB to SSPx:  {'SSP1-PkBudg500': 'ssp119', 'SSP2-Base': 'ssp245', 'SSP5-Base': 'ssp585'}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
pGWP100 are imported as:  [('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP585_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP119_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021

In [3]:
from utils_pGWP.pGWP_F2 import *

In [4]:
bw2data.projects.set_current('iveo_v1')
mybio = bw2data.Database("biosphere3")
initProject("iveo_v1")

Biosphere database already present!!! No setup is needed


In [5]:
print( ssp_scn  , ssp_yr) 
bw2_method_metric = 'pGWP' #'dpIRF'
root_path = '/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/'
tech_p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/p_tech_perSSP_Y.xlsx'))
consmp_p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/tech_conversion_veh_p/p_scn_v_need_tech.xlsx'))

['_remind_SSP1-PkBudg500', '_remind_SSP2-Base', '_remind_SSP5-Base'] [2030, 2040, 2050]


### A. init set up v_lifetime and annual mileage¶

In [6]:
pdb = bw.Database("iveo_Parameterized_v1")
print(len(pdb))
agb.loadParams()

384


[ParamRegistry] Param NMC622_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param NCA_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LFP_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LTO_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_NG_MJperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_elec_kWhperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param g_truck_vehicle_prod_lightweighting was already defined in '<project>' : overriding.
[ParamRegistry] Param battery_total_capacity_for_gtruck was already defined in '<project>' : overriding.
[ParamRegistry] Param biodiesel_UCO_perc was already defined in '<project>' : overriding.
[ParamRegistry] Pa

In [7]:
v_lifetime = 20 
v_yearlyFU = 36000 #annual mileage [km] could be either a float (same yearly FU), or a list e.g., [12000] * v_lifetime 

#### A.1 get UPR

In [8]:
gtruck_finalelec_switch_allsspyear = [] 
for prm_name in ssp_scn:
    for year_t in ssp_yr: 
        bg_db_ssp_yr = bg_ei_ + prm_name + "_" + str(year_t)            # backgroun database (BD) to use for SSP[x], Year[t]
        copied_prm_UPR_suffix = "_from_" + prm_name + "_" + str(year_t) # copy UPR from BD to PDB for further use/modification if needed
        newUPR_suffix =  "_wh_" + prm_name + "_" + str(year_t)          # naming on any new UPRs created in PDB
        act = agb.findActivity( db_name = PDB_NAME, name="elec_tech_switch_act" + newUPR_suffix )
        gtruck_finalelec_switch_allsspyear.append(act)

len(gtruck_finalelec_switch_allsspyear)

9

### A.2 read in TRL table to see which switch_P to apply

In [9]:
data = xr.open_dataset( os.path.abspath(os.path.join(root_path, 'data/to_use_interm/TRL_1C_vehDSwhTRL.nc')))
data2 = data.where(data["tech_avail"] == 1, drop = True) 
data2

<xarray.Dataset>
Dimensions:     (v_type: 1, size: 1, powertrain: 2, v_year: 3, SSP: 4,
                 Module: 5, Tech: 9)
Coordinates:
  * Tech        (Tech) object 'LIB_LFP' 'LIB_LTO' ... 'dummy_tech' 'grid_elec'
  * Module      (Module) object 'F2_fuel_diesel' 'F2_fuel_elec' ... 'V2_oper'
  * SSP         (SSP) object 'ssp119' 'ssp126' 'ssp245' 'ssp585'
  * v_year      (v_year) int32 2030 2040 2050
  * powertrain  (powertrain) object 'BEV' 'ICEV-d'
  * v_type      (v_type) object 'garbage_truck'
  * size        (size) object 'HDV_class7'
Data variables:
    TRL         (v_type, size, powertrain, v_year, SSP, Module, Tech) float64 ...
    tech_appl   (v_type, size, powertrain, v_year, SSP, Module, Tech) float64 ...
    tech_avail  (v_type, size, powertrain, v_year, SSP, Module, Tech) float64 ...

### B. testing read   consmp_p file

In [10]:
consmp_p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/tech_conversion_veh_p/p_scn_v_need_tech.xlsx'))

for act_tocalc in gtruck_finalelec_switch_allsspyear:
    
    str1 = act_tocalc.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)

    SSP_vY_consmp_list = [] 
    v_pneed = loadYearlyParamswhsize(PARAMS_FILE = consmp_p_file, s_name = 'g_truck', hypothesis = 'point value', 
                               years = [year_to_get_inP], SSP = ssp_to_get_inP, size = 'HDV_class7')
    """ final run year should be v_lifetime, if assuming it starts running from YEAR[0], then no need to adding (v_lifetime+1) in range() """ 
    for Y in range(year_to_get_inP+1, year_to_get_inP + v_lifetime + 1,  1):  
        get_f2_fuel_elec_kwh_diesel_kg(powertrain = "BEV", Y = Y, v_lifetime = 20, v_yearlyFU = v_yearlyFU, 
                                       SP = ssp_to_get_inP, year_inP = year_to_get_inP, v_pneed = v_pneed, to_print_consmpt = True )
    

  ⛽️ 🔌 for v_year2030, running_year 2031, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2032, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2033, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2034, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2035, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2036, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2037, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2038, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2039, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2040, SSP ssp119, the annual fuel_consump is 80370.0 kWh 
  ⛽️ 🔌 for v_year2030, running_year 2041, SSP ssp119, the an

## C. calc

DO NOT run C.1 static inventory, as shown in F2_diesel module, not as accurate as dynamic inventory 
### C.2 run calculation annually, dynamic inventory each year

In [15]:

GWP_list, SSP_list, Year_list, tech_list, run_Y_list  = [], [],[], [], []

""" first loop is for each v_year:: [year_to_get_inP] and  SSP:: [ssp_to_get_inP]""" 
for act_tocalc in gtruck_finalelec_switch_allsspyear:
    # need a matched YearlyParams for each SSPx:
    str1 = act_tocalc.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)
    #print(ssp_to_get_inP, year_to_get_inP)
 
    """ prepare methods: GWP1_list using static GWP,  GWP2_list using the pGWP100 for the matched v_year """ 
    gwp_spx_yt = get_my_pGWP100(ssp = ssp_to_get_inP, my = year_to_get_inP, metric = 'pGWP') 
    methods_touse = [chosen_methods[0], chosen_methods[1], gwp_spx_yt] # IPCCGWP100, premiseGWP, pGWP100_per_premiseGWP


    """ see what tech can be used under SSPx, Year[t], should move TRL into each Y loop then  """
    for Y in range(year_to_get_inP + 1, year_to_get_inP + v_lifetime + 1,  1):  
        run_Y_GWP_list,run_Y_SSP_list, run_Y_Year_list, run_Y_tech_list, run_Y = [],[],[],[],[] 
        
        """ we read in P_files for each year, we have values for each five year gap, more detailed than the v_year per 10 years, """ 
        yr_toget = min([2030, 2035, 2040, 2045, 2050], key=lambda x: abs(x - Y))
        """ no tech_P file for elec """
        v_pneed = loadYearlyParamswhsize(PARAMS_FILE = consmp_p_file, s_name = 'g_truck', hypothesis = 'point value', 
                               years = [year_to_get_inP], SSP = ssp_to_get_inP, size = 'HDV_class7')

        #print(f"for v_year {year_to_get_inP} and running {Y}, get cloest year from P_files {yr_toget} " )
        
        act_amount = get_f2_fuel_elec_kwh_diesel_kg(powertrain = "BEV", Y = Y, v_lifetime = 20, v_yearlyFU = v_yearlyFU, 
                                                    SP = ssp_to_get_inP,  year_inP = year_to_get_inP, v_pneed = v_pneed)

        """ v_year = yr_toget_4TRL_10gap when reading TRL for tech related to running year, not year_to_get_inP as v_year """ 
        """ as we don't have TRL per 5 years prepared like for tech / consumption_P files """ 
        yr_toget_4TRL_10gap = min([2030, 2040, 2050], key=lambda x: abs(x - Y))
        all_app_tech = []  
        for tech_x in ["REC_100PV", "grid_elec"]:  
            avail = data2.sel(Tech = tech_x, powertrain = "BEV", Module = "F2_fuel_elec", v_type = "garbage_truck",  
                                  SSP = ssp_to_get_inP, v_year = yr_toget_4TRL_10gap )["tech_avail"].values
            if np.isnan(avail).all(): 
                pass
            else: 
                all_app_tech.append(tech_x)
            #print(f"for v_year {year_to_get_inP} and running {Y}, reading per TRL_year{yr_toget_4TRL_10gap}, tech avaialbe: {all_app_tech} " )
        
        for e_tech in all_app_tech :
            if e_tech == "REC_100PV":      #### mappig names in swtich_P  "TechUPR_F2_elec_whSwitchP.ipynb" 
                e_tech = "onsite100PV"
            elif e_tech == "grid_elec":
                e_tech = "grid"
            else:
                pass 
            
            df = agb.multiLCAAlgebric(models=[
                    (act_tocalc, act_amount)],  #act_test
                    methods = methods_touse,   
                    #define switch parameter here if you have one, otherwise, using the default value, e.g., assuming assembly @CA_QC 
                    gtruck_BEV_elec_tech = e_tech ,
                    #**yearlyParams  #we don't have tech_P for elec, currently just relying on premise BGDB 
                )
            print(f"end of running LCA calc for SSP {ssp_to_get_inP} and year {year_to_get_inP} using elec_prod tech {e_tech} for {act_tocalc} ")
            #display(df)
        
            lcia_allyears = [] 
            for t in range(len(df.columns)): 
                lcia_t = df.iloc[:,t].values[0]
                lcia_allyears.append(lcia_t)
            
            run_Y_GWP_list.append(lcia_allyears)
            run_Y_SSP_list.append(ssp_to_get_inP) 
            run_Y_Year_list.append(year_to_get_inP)
            run_Y_tech_list.append(e_tech)
            run_Y.append(Y)
            #print(len(run_Y_GWP_list),len(run_Y_SSP_list),len(run_Y_Year_list),len(run_Y_tech_list),len(run_Y))
        
        GWP_list.append(run_Y_GWP_list)
        SSP_list.append(run_Y_SSP_list) 
        Year_list.append(run_Y_Year_list)
        tech_list.append(run_Y_tech_list)        
        run_Y_list.append(run_Y) 
        

end of running LCA calc for SSP ssp119 and year 2030 using elec_prod tech grid for 'elec_tech_switch_act_wh__remind_SSP1-PkBudg500_2030' (kilowatt hour, GLO, None) 
end of running LCA calc for SSP ssp119 and year 2030 using elec_prod tech grid for 'elec_tech_switch_act_wh__remind_SSP1-PkBudg500_2030' (kilowatt hour, GLO, None) 
end of running LCA calc for SSP ssp119 and year 2030 using elec_prod tech grid for 'elec_tech_switch_act_wh__remind_SSP1-PkBudg500_2030' (kilowatt hour, GLO, None) 
end of running LCA calc for SSP ssp119 and year 2030 using elec_prod tech grid for 'elec_tech_switch_act_wh__remind_SSP1-PkBudg500_2030' (kilowatt hour, GLO, None) 
end of running LCA calc for SSP ssp119 and year 2030 using elec_prod tech grid for 'elec_tech_switch_act_wh__remind_SSP1-PkBudg500_2030' (kilowatt hour, GLO, None) 
end of running LCA calc for SSP ssp119 and year 2030 using elec_prod tech onsite100PV for 'elec_tech_switch_act_wh__remind_SSP1-PkBudg500_2030' (kilowatt hour, GLO, None) 
end

In [29]:
d2 = get_plcia_wh_v_mod_bev_grid(SSP_list,run_Y_list, tech_list, Year_list, GWP_list,  v_module = "F2_fuel_elec", lciacol_names = ['IPCC2021_default', "premise_GWP",  'pGWP100' ] )[1]
d2

IPCC2021_default   premise_GWP  \
SSP    v_module     v_year tech_list run_year                                   
ssp119 F2_fuel_elec 2030   grid      2031           2915.047506   1514.289193   
                                     2032           2915.047506   1514.289193   
                                     2033           2915.047506   1514.289193   
                                     2034           2915.047506   1514.289193   
                                     2035           2915.047506   1514.289193   
...                                                         ...           ...   
ssp585 F2_fuel_elec 2050   grid      2066          22672.972813  22711.723215   
                                     2067          22672.972813  22711.723215   
                                     2068          22672.972813  22711.723215   
                                     2069          22672.972813  22711.723215   
                                     2070          22672.972813  22711.723215   

                                                    pGWP100  
SSP    v_module     v_year tech_list run_year                
ssp119 F2_fuel_elec 2030   grid      2031       1466.885077  
                                     2032       1466.885077  
                                     2033       1466.885077  
                                     2034       1466.885077  
                                     2035       1466.885077  
...                                                     ...  
ssp585 F2_fuel_elec 2050   grid      2066      22385.024298  
                                     2067      22385.024298  
                                     2068      22385.024298  
                                     2069      22385.024298  
                                     2070      22385.024298  

[180 rows x 3 columns]

In [35]:
d3 = get_plcia_wh_v_mod_bev_grid(SSP_list,run_Y_list, tech_list, Year_list, GWP_list,  v_module = "F2_fuel_elec", lciacol_names = ['IPCC2021_default', "premise_GWP",  'pGWP100' ] )[2]
d3

,IPCC2021_default,premise_GWP,pGWP100,SSP,v_year,tech_list,run_year,v_module
0,2915.047506,1514.289193,1466.885077,ssp119,2030,grid,2031,F2_fuel_elec
1,2915.047506,1514.289193,1466.885077,ssp119,2030,grid,2032,F2_fuel_elec
2,2915.047506,1514.289193,1466.885077,ssp119,2030,grid,2033,F2_fuel_elec
3,2915.047506,1514.289193,1466.885077,ssp119,2030,grid,2034,F2_fuel_elec
4,2915.047506,1514.289193,1466.885077,ssp119,2030,grid,2035,F2_fuel_elec
...,...,...,...,...,...,...,...,...
325,22672.972813,22711.723215,22385.024298,ssp585,2050,grid,2068,F2_fuel_elec
326,2754.270196,2768.915661,2724.016288,ssp585,2050,onsite100PV,2069,F2_fuel_elec
327,22672.972813,22711.723215,22385.024298,ssp585,2050,grid,2069,F2_fuel_elec
328,2754.270196,2768.915661,2724.016288,ssp585,2050,onsite100PV,2070,F2_fuel_elec


In [31]:
d1 =  get_plcia_wh_v_mod_bev_grid(SSP_list,run_Y_list, tech_list, Year_list, GWP_list,  v_module = "F2_fuel_elec", lciacol_names = ['IPCC2021_default', "premise_GWP",  'pGWP100' ] )[0]
d1

IPCC2021_default    premise_GWP  \
SSP    v_module     v_year tech_list                                    
ssp119 F2_fuel_elec 2030   grid           58300.950124   30285.783854   
                    2040   grid           35452.746293    6807.204688   
                    2050   grid           31822.488225   10006.346013   
ssp245 F2_fuel_elec 2030   grid          341629.606047  329998.324107   
                    2040   grid          177347.166000  171761.261522   
                    2050   grid           76088.165606   76174.968621   
ssp585 F2_fuel_elec 2030   grid          398019.861183  386423.074807   
                    2040   grid          448501.449088  443346.944189   
                    2050   grid          453459.456263  454234.464293   

                                            pGWP100  
SSP    v_module     v_year tech_list                 
ssp119 F2_fuel_elec 2030   grid        29337.701540  
                    2040   grid         6377.565420  
                    2050   grid         9669.330577  
ssp245 F2_fuel_elec 2030   grid       325671.695229  
                    2040   grid       170015.279945  
                    2050   grid        75600.105820  
ssp585 F2_fuel_elec 2030   grid       378360.595163  
                    2040   grid       434058.017847  
                    2050   grid       447700.485951

In [32]:
bev_f2_ds = get_plcia_ds(d1)
bev_f2_ds

<xarray.Dataset>
Dimensions:          (SSP: 3, v_module: 1, v_year: 3, tech_list: 1,
                      Impact_category: 3)
Coordinates:
  * SSP              (SSP) object 'ssp119' 'ssp245' 'ssp585'
  * v_module         (v_module) object 'F2_fuel_elec'
  * v_year           (v_year) int64 2030 2040 2050
  * tech_list        (tech_list) object 'grid'
  * Impact_category  (Impact_category) object 'IPCC2021_default' ... 'premise...
Data variables:
    value            (SSP, v_module, v_year, tech_list, Impact_category) float64 ...

In [33]:
bev_f2_ds.to_netcdf('pGWP_module_output/F2_BEVGT_pGWP100.nc')

In [36]:
d1.to_excel('pGWP_module_output/excel_output/F2_BEVGT_pGWP100.xlsx' )
d2.to_excel('pGWP_module_output/excel_output/F2_BEVGT_pGWP100_annual_gridonly.xlsx' )
d3.to_excel('pGWP_module_output/excel_output/F2_BEVGT_pGWP100_annual_grid_orPV.xlsx' )